In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai",
    "sentence-transformers==2.7.0", "timm==0.9.16",
    "faiss-cpu", "requests", "tf-keras",
    "--quiet", "--no-deps", "--upgrade"])
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM -- BCP -- Colab 01: Condor RAG Agent
# ============================================================
import os, sys, json, random, re
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import os, sys

# Ensure agent_base and agent_server are importable from this directory
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.abspath(".."))

# Load credentials from ~/.env_coliseum
_env = os.path.expanduser("~/.env_coliseum")
if os.path.exists(_env):
    for _l in open(_env):
        _l = _l.strip()
        if _l and "=" in _l:
            _k, _v = _l.split("=", 1)
            os.environ[_k] = _v
    print("Credentials loaded from ~/.env_coliseum")
else:
    print("WARNING: ~/.env_coliseum not found")

AZURE_API_KEY  = os.environ.get("AZURE_API_KEY",  "")
AZURE_BASE_URL = os.environ.get("AZURE_BASE_URL", "https://rsgd15-foundry.openai.azure.com/openai/v1/")
MODEL          = "gpt-5"
ARENA_URL      = "https://agent-coliseum.onrender.com"
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "")

print(f"AZURE_API_KEY:  ...{AZURE_API_KEY[-6:] if AZURE_API_KEY else 'NOT SET'}")
print(f"AZURE_BASE_URL: {AZURE_BASE_URL}")
print(f"NGROK_TOKEN:    ...{NGROK_TOKEN[-6:] if NGROK_TOKEN else 'NOT SET'}")

from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

def build_rag_index(facts_path="latam_facts.jsonl"):
    # Try current dir and parent dir
    if not os.path.exists(facts_path):
        facts_path = os.path.join("..", "data", "latam_facts.jsonl")
    facts = []
    with open(facts_path) as f:
        for line in f:
            if line.strip():
                facts.append(json.loads(line))
    texts      = [f["text"] for f in facts]
    vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1,2),
                                 sublinear_tf=True, min_df=1)
    matrix = vectorizer.fit_transform(texts)
    print(f"RAG index ready: {len(facts)} facts, {matrix.shape[1]} TF-IDF features")
    return vectorizer, matrix, facts

def search_rag(query: str, top_k: int = 3) -> list:
    vec    = rag_vectorizer.transform([query])
    scores = cosine_similarity(vec, rag_matrix).flatten()
    top    = np.argsort(scores)[::-1][:top_k]
    return [rag_facts[i]["text"] for i in top if scores[i] > 0]

rag_vectorizer, rag_matrix, rag_facts = build_rag_index()

class CondorAgent(Agent):
    name        = "Condor RAG"
    avatar      = "🦅"
    description = "Agente con memoria y busqueda semantica de hechos latinoamericanos"

    def __init__(self):
        self._memory = {}

    def on_arena_start(self, ctx: WorldContext) -> None:
        self._memory = {}

    def on_match_start(self, ctx: MatchContext) -> None:
        if ctx.opponent_agent_id not in self._memory:
            self._memory[ctx.opponent_agent_id] = {"name": ctx.opponent_name, "wins": 0, "losses": 0}

    def on_match_end(self, ctx: MatchContext, result: MatchResult) -> None:
        opp = ctx.opponent_agent_id
        won = result.winner_id == ctx.my_agent_id
        if opp in self._memory:
            if won: self._memory[opp]["wins"]   += 1
            else:   self._memory[opp]["losses"] += 1

    def move(self, ctx: WorldContext) -> Position:
        active = [a for a in ctx.agents
                  if a.status == "active" and a.agent_id != ctx.my_agent_id]
        if not active:
            dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0)])
            return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                            y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))
        target = min(active, key=lambda a: a.score)
        dx = 1 if target.position.x > ctx.my_position.x else -1 if target.position.x < ctx.my_position.x else 0
        dy = 1 if target.position.y > ctx.my_position.y else -1 if target.position.y < ctx.my_position.y else 0
        return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                        y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))

    def think(self, ctx: MatchContext) -> str:
        rag_hits    = search_rag(ctx.topic + " " + ctx.current_question, top_k=3)
        rag_context = "\n".join(f"- {f}" for f in rag_hits) or "(no relevant facts found)"
        opp_mem     = self._memory.get(ctx.opponent_agent_id, {})
        opp_summary = (f"Wins: {opp_mem.get('wins',0)}, Losses: {opp_mem.get('losses',0)}"
                       if opp_mem else "No prior history.")
        history_text = ""
        for t in ctx.history[-3:]:
            history_text += f"\n  Turn {t['turn_number']}: Q={t['question'][:60]} A={t['answer'][:60]} Score={t['score']}"

        role_instruction = (
            f"Generate a sharp, specific question about {ctx.topic} that will be hard for {ctx.opponent_name} to answer."
            if ctx.role == "asker" else
            f"Answer this question accurately in 1-2 sentences: {ctx.current_question}"
        )

        prompt = f"""You are competing in a Latin America knowledge tournament.
Topic: {ctx.topic} | Role: {ctx.role} | Turn: {ctx.turn}/{ctx.total_turns}
Your score: {sum(ctx.my_scores)} pts | Opponent: {sum(ctx.opponent_scores)} pts
Opponent: {ctx.opponent_name} | Profile: {opp_summary}
History:{history_text if history_text else " (first turn)"}
Knowledge base:
{rag_context}

Task: {role_instruction}

Reason step by step. Write concrete responses (do NOT leave blanks):

SITUATION [Wei et al. 2022 -- Chain-of-Thought]:
OPPONENT [Langner et al. 2023 -- Theory of Mind]:
GOAL [Yao et al. 2022 -- ReAct]:
DRAFT [Nye et al. 2021 -- Scratchpad]:
CRITIQUE [Madaan et al. 2023 -- Self-Refine]:
FINAL [Shinn et al. 2023 -- Reflexion]: <your {'question' if ctx.role == 'asker' else 'answer'}, concise>"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=2000,
        )
        return response.choices[0].message.content.strip()

    def ask(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        import re
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL"):
                after = clean[5:].lstrip(":").strip()
                after = re.sub(r"^\[.*?\]\s*:?\s*", "", after).strip()
                if after:
                    return after
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

import threading, requests as _req, time as _time

def _keepalive(arena_url):
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

agent = CondorAgent()

In [ ]:
# ── DEBUG CELL — test agent before connecting to arena ────────────────────
import requests, json as _json
from agent_base import MatchContext

# ── Test 1: Flask health ──────────────────────────────────────────────────
try:
    r = requests.get("http://localhost:5000/health", timeout=3)
    print("Flask health:", r.status_code, r.json())
except Exception as e:
    print("Flask NOT reachable:", e)

# ── Test 2: ngrok tunnel ──────────────────────────────────────────────────
try:
    r = requests.get("http://localhost:4040/api/tunnels", timeout=3)
    tunnels = r.json().get("tunnels", [])
    for t in tunnels:
        print("Ngrok tunnel:", t["public_url"], "->", t["config"]["addr"])
    if not tunnels:
        print("No active ngrok tunnels")
except Exception as e:
    print("Ngrok NOT reachable:", e)

# ── Test 3: raw think() output ────────────────────────────────────────────
test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)
print("\n=== RAW think() OUTPUT (asker) ===")
raw = agent.think(test_ctx)
print(raw)
print("\n=== EXTRACTED FINAL (asker) ===")
print(repr(agent._extract_final(raw)))

test_ctx2 = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)
print("\n=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx2)
print(raw2)
print("\n=== EXTRACTED FINAL (answerer) ===")
print(repr(agent._extract_final(raw2)))

# ── Test 4: simulate /ask and /answer HTTP calls ──────────────────────────
payload = {
    "match_id": "test", "topic": "Latin American geography",
    "turn": 1, "total_turns": 3, "role": "asker",
    "history": [], "my_agent_id": "me", "opponent_agent_id": "opp",
    "opponent_name": "Test", "my_scores": [], "opponent_scores": [],
    "scratchpad": "", "current_question": ""
}
try:
    r = requests.post("http://localhost:5000/ask", json=payload, timeout=60)
    print("\n=== /ask HTTP response ===")
    print("Status:", r.status_code)
    print("Body:", _json.dumps(r.json(), indent=2))
except Exception as e:
    print("\n/ask call failed:", e)

payload["role"] = "answerer"
payload["current_question"] = "What is the longest river in South America?"
try:
    r = requests.post("http://localhost:5000/answer", json=payload, timeout=60)
    print("\n=== /answer HTTP response ===")
    print("Status:", r.status_code)
    print("Body:", _json.dumps(r.json(), indent=2))
except Exception as e:
    print("\n/answer call failed:", e)

In [ ]:
import subprocess
subprocess.run(["fuser", "-k", "5000/tcp"], capture_output=True)

from pyngrok import ngrok
ngrok.kill()

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5000,
    ngrok_token = NGROK_TOKEN,
)